In [13]:
from utils.DataPreprocessing import dataloader
from loguru import logger
import torch
import os

path = os.path.dirname(os.path.dirname(os.getcwd()))
print(path)
data = dataloader(path=path)

2024-04-06 17:12:47.624 | INFO     | utils.DataPreprocessing:load_csv:123 - shape: (11_103, 3)
┌───────────┬─────────────┬────────────┐
│ AVG_TEMP  ┆ AVG_TEMP_DC ┆ date       │
│ ---       ┆ ---         ┆ ---        │
│ f32       ┆ str         ┆ date       │
╞═══════════╪═════════════╪════════════╡
│ 29.299999 ┆ C           ┆ 1992-07-01 │
│ 29.200001 ┆ C           ┆ 1992-07-02 │
│ 29.6      ┆ C           ┆ 1992-07-03 │
│ 29.299999 ┆ C           ┆ 1992-07-04 │
│ 29.299999 ┆ C           ┆ 1992-07-05 │
│ …         ┆ …           ┆ …          │
│ 22.200001 ┆ C           ┆ 2022-11-26 │
│ 22.4      ┆ C           ┆ 2022-11-27 │
│ 25.4      ┆ C           ┆ 2022-11-28 │
│ 25.1      ┆ C           ┆ 2022-11-29 │
│ 22.1      ┆ C           ┆ 2022-11-30 │
└───────────┴─────────────┴────────────┘
2024-04-06 17:12:47.625 | INFO     | utils.DataPreprocessing:load_csv:125 - shape: (1, 3)
┌──────────┬─────────────┬──────┐
│ AVG_TEMP ┆ AVG_TEMP_DC ┆ date │
│ ---      ┆ ---         ┆ ---  │
│ u32      ┆ u32

/home/argonaut/programming/HK-Temperature-Forecasting


In [15]:
logger.success(data.X_train)
logger.success(data.y_train)
logger.success(data.X_test)
logger.success(data.y_test)

2024-04-06 17:12:50.758 | SUCCESS  | __main__:<module>:1 - shape: (5_441, 34)
┌───────────┬─────┬──────┬─────┬───┬──────────────┬──────────────┬──────────────┬──────────────┐
│ GSR       ┆ SUN ┆ RH   ┆ UV  ┆ … ┆ AVG_TEMP_t-4 ┆ AVG_TEMP_t-3 ┆ AVG_TEMP_t-2 ┆ AVG_TEMP_t-1 │
│ ---       ┆ --- ┆ ---  ┆ --- ┆   ┆ ---          ┆ ---          ┆ ---          ┆ ---          │
│ f32       ┆ f32 ┆ f32  ┆ f32 ┆   ┆ f32          ┆ f32          ┆ f32          ┆ f32          │
╞═══════════╪═════╪══════╪═════╪═══╪══════════════╪══════════════╪══════════════╪══════════════╡
│ 16.440001 ┆ 6.9 ┆ 75.0 ┆ 4.0 ┆ … ┆ 28.799999    ┆ 29.0         ┆ 29.299999    ┆ 30.299999    │
│ 1.93      ┆ 0.0 ┆ 79.0 ┆ 0.6 ┆ … ┆ 29.0         ┆ 29.299999    ┆ 30.299999    ┆ 29.299999    │
│ 3.04      ┆ 0.0 ┆ 92.0 ┆ 1.0 ┆ … ┆ 29.299999    ┆ 30.299999    ┆ 29.299999    ┆ 25.9         │
│ 4.34      ┆ 0.0 ┆ 91.0 ┆ 1.0 ┆ … ┆ 30.299999    ┆ 29.299999    ┆ 25.9         ┆ 24.1         │
│ 2.04      ┆ 0.0 ┆ 93.0 ┆ 0.5 ┆ … ┆ 29.299999   

In [16]:
batch_size = 32
epochs = 100 

# Get cpu or gpu device for training.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [42]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert Polars DataFrame and Series to NumPy arrays
X_train = data.X_train.to_numpy()
y_train = data.y_train.to_numpy()
X_val = data.X_val.to_numpy()
y_val = data.y_val.to_numpy()

# Convert NumPy arrays to PyTorch Tensors
X_train_tensor = torch.from_numpy(X_train).unsqueeze(1)
y_train_tensor = torch.from_numpy(y_train).unsqueeze(1)
X_val_tensor = torch.from_numpy(X_val).unsqueeze(1)
y_val_tensor = torch.from_numpy(y_val).unsqueeze(1)

# Create TensorDatasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


In [51]:
for x, y in train_loader:
    print(x.shape, y.shape)
    break

torch.Size([32, 1, 34]) torch.Size([32, 1])


In [52]:
from model import PatchTST

c_in = 34 # number of features
context_window = 20 # number of time steps from the past that the model will look at when making a prediction.
target_window = 1
patch_len = 3
stride = 2 

model = PatchTST(c_in=c_in, 
                context_window=context_window, 
                target_window=target_window, 
                patch_len=patch_len, 
                stride=stride)

model = model.to(device)
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
# optimizer = torch.optim.SGD(model.parameters(), lr=0.02, momentum=0.9, weight_decay=5e-4)
loss_fn = torch.nn.MSELoss()
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, mode="min", factor=0.5, min_lr=0, patience=2, verbose=True
# )
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=250, gamma=0.5, verbose=False)

PatchTST(
  (revin_layer): RevIN()
  (padding_patch_layer): ReplicationPad1d((0, 2))
  (backbone): TSTiEncoder(
    (W_P): Linear(in_features=3, out_features=16, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
    (encoder): TSTEncoder(
      (layers): ModuleList(
        (0-2): 3 x TSTEncoderLayer(
          (self_attn): _MultiheadAttention(
            (W_Q): Linear(in_features=16, out_features=16, bias=True)
            (W_K): Linear(in_features=16, out_features=16, bias=True)
            (W_V): Linear(in_features=16, out_features=16, bias=True)
            (sdp_attn): _ScaledDotProductAttention(
              (attn_dropout): Dropout(p=0.0, inplace=False)
            )
            (to_out): Sequential(
              (0): Linear(in_features=16, out_features=16, bias=True)
              (1): Dropout(p=0.3, inplace=False)
            )
          )
          (dropout_attn): Dropout(p=0.3, inplace=False)
          (norm_attn): Sequential(
            (0): Transpose()
            

In [54]:
from torch.cuda.amp import autocast as autocast
from torch.cuda.amp import GradScaler
from tqdm import tqdm
# Creates a GradScaler once at the beginning of training.
scaler = GradScaler()

# Training function
def train(dataloader, model, loss_fn, optimizer):

    # Turn on training mode
    model.train()

    train_loss = 0

    for X, y in tqdm(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error

        # print(X.shape, y.shape)
        # pred = model(X)
        # loss = loss_fn(pred, y)

        with autocast():
            pred = model(X)
            # output is float16 because linear layers autocast to float16.
            assert pred.dtype is torch.float16

            loss = loss_fn(pred, y)
            # loss is float32 because mse_loss layers autocast to float32.
            assert loss.dtype is torch.float32

        # Backpropagation
        optimizer.zero_grad()
        # Scales loss.  Calls backward() on scaled loss to create scaled gradients.
        # Backward passes under autocast are not recommended.
        # Backward ops run in the same dtype autocast chose for corresponding forward ops.
        scaler.scale(loss).backward()

        # scaler.step() first unscales the gradients of the optimizer's assigned params.
        # If these gradients do not contain infs or NaNs, optimizer.step() is then called,
        # otherwise, optimizer.step() is skipped.
        scaler.step(optimizer)

        # Updates the scale for next iteration.
        scaler.update()

        # record loss
        train_loss += loss.item()

    train_loss /= len(dataloader)

    print(f"MSE loss: {train_loss:>8f}")
    return train_loss

In [55]:
# val function
def val(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    # Turn on evalution mode
    model.eval()
    val_loss = 0

    # Turn off gradient descent
    with torch.no_grad():
        for X, y in tqdm(dataloader):
            X, y = X.to(device), y.to(device)
            pred = model(X)

            # record loss
            val_loss += loss_fn(pred, y).item()

    val_loss /= num_batches

    print(f"MSE loss: {val_loss:>8f}")
    return val_loss

In [56]:
train_losses = []
val_losses = []
learning_rate = []

In [57]:
import numpy as np
# Total training epochs
epochs = 50

es_track_loss = np.inf
patience = 3
trigger_count = 0

for t in range(epochs):
    print('\n', "=" * 25, "Epoch", t + 1, "=" * 25)
    train_loss = train(train_loader, model, loss_fn, optimizer)
    val_loss = val(val_loader, model, loss_fn)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    # scheduler.step(test_loss)
    learning_rate.append(optimizer.param_groups[0]['lr'])
    if val_loss > es_track_loss:
        trigger_count += 1
    else:
        trigger_count = 0
    if trigger_count > patience:
        print(f'Early Stopped at {t} Epoch')
        break
    es_track_loss = val_loss

print(" Done!")


 ========================= Epoch 1 =========================


  0%|          | 0/171 [00:00<?, ?it/s]


RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Half